IMPORT LIBRARIES

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


LOAD DATASET

In [37]:
df=pd.read_csv("online_retail_Dataset.csv")
print("Dataset loaded successfully!")

Dataset loaded successfully!


EXPLORE DATSET

In [38]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [39]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 1067371
Number of columns: 8


In [40]:
df.columns.tolist()

['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'Customer ID',
 'Country']

In [41]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB


MISSING VALUES

In [43]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

DUPLICATE RECORDS

In [44]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 34335


DATA CLEANING

In [45]:
# Create a copy of the original dataset
clean_df=df.copy()

print("Original dataset shape:", df.shape)
print("Working dataset shape:", clean_df.shape)

Original dataset shape: (1067371, 8)
Working dataset shape: (1067371, 8)


CHECK MISSING VALUES

In [46]:
# Check missing values
missing_values = clean_df.isnull().sum()

missing_summary = pd.DataFrame({
    'Column': clean_df.columns,
    'Missing Values': missing_values.values,
    'Missing Percentage': (
        missing_values.values / len(clean_df)
    ) * 100
})

missing_summary

,Column,Missing Values,Missing Percentage
0,Invoice,0,0.000000
1,StockCode,0,0.000000
2,Description,4382,0.410541
3,Quantity,0,0.000000
4,InvoiceDate,0,0.000000
5,Price,0,0.000000
6,Customer ID,243007,22.766873
7,Country,0,0.000000


REMOVE MISSING CUSTOMER ID'S

In [47]:
# Remove records with missing Customer ID
clean_df = clean_df.dropna(subset=['Customer ID'])

print("Shape after removing missing Customer IDs:", clean_df.shape)

print("\nRemaining missing values:")
print(clean_df.isnull().sum())

Shape after removing missing Customer IDs: (824364, 8)

Remaining missing values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


REMOVE DUPLICATE RECORDS

In [48]:
# Count duplicates before removal
print("Duplicates before cleaning:", clean_df.duplicated().sum())

# Remove duplicates
clean_df = clean_df.drop_duplicates()

print("Duplicates after cleaning:", clean_df.duplicated().sum())
print("Shape after removing duplicates:", clean_df.shape)

Duplicates before cleaning: 26479
Duplicates after cleaning: 0
Shape after removing duplicates: (797885, 8)


CONVERTING INVOICE DATA INTO DATETIME FORMAT

In [49]:
# Convert InvoiceDate to datetime
clean_df['InvoiceDate'] = pd.to_datetime(
    clean_df['InvoiceDate'],
    errors='coerce'
)

# Check data type
print(clean_df['InvoiceDate'].dtype)

# Check invalid dates
print("Invalid dates:", clean_df['InvoiceDate'].isnull().sum())

datetime64[us]
Invalid dates: 0


REMOVE CANCELLED TRANSACTIONS

In [50]:
# Remove cancelled invoices
clean_df = clean_df[
    ~clean_df['Invoice'].astype(str).str.startswith('C')
].copy()

print("Shape after removing cancelled transactions:", clean_df.shape)

Shape after removing cancelled transactions: (779495, 8)


REMOVE INVALID QUANTITY AND PRICE VALUES

In [51]:
# Keep only positive quantities and prices
clean_df = clean_df[
    (clean_df['Quantity'] > 0) &
    (clean_df['Price'] > 0)
].copy()

print("Shape after removing invalid Quantity and Price:", clean_df.shape)

Shape after removing invalid Quantity and Price: (779425, 8)


CREATE REVENUE COLUMN

In [52]:
# Calculate revenue
clean_df['Revenue'] = clean_df['Quantity'] * clean_df['Price']

clean_df[['Invoice', 'Quantity', 'Price', 'Revenue']].head(10)

,Invoice,Quantity,Price,Revenue
0,489434,12,6.95,83.4
1,489434,12,6.75,81.0
2,489434,12,6.75,81.0
3,489434,48,2.10,100.8
4,489434,24,1.25,30.0
5,489434,24,1.65,39.6
6,489434,24,1.25,30.0
7,489434,10,5.95,59.5
8,489435,12,2.55,30.6
9,489435,12,3.75,45.0


CONVERT CUSTOMER ID INTO INTEGER

In [53]:
# Convert Customer ID from float to integer
clean_df['Customer ID'] = clean_df['Customer ID'].astype('int64')

print(clean_df['Customer ID'].dtype)

int64


FINAL DATASET CHECK

In [54]:
print("FINAL CLEANED DATASET")

print("\nDataset shape:", clean_df.shape)

print("\nMissing values:")
print(clean_df.isnull().sum())

print("\nDuplicate records:", clean_df.duplicated().sum())

print("\nData types:")
print(clean_df.dtypes)

print("\nFirst five cleaned records:")
display(clean_df.head())

FINAL CLEANED DATASET

Dataset shape: (779425, 9)

Missing values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
dtype: int64



Duplicate records: 0

Data types:
Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
Revenue               float64
dtype: object

First five cleaned records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


SAVING THE CLEANED DATASET

In [55]:
clean_df.to_csv("cleaned_online_retail.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


TOTAL BUSSINESS OVERVIEW

In [56]:
print("Total Transactions:", clean_df['Invoice'].nunique())

print("Total Customers:", clean_df['Customer ID'].nunique())

print("Total Products:", clean_df['StockCode'].nunique())

print("Total Countries:", clean_df['Country'].nunique())

print("Total Revenue:", round(clean_df['Revenue'].sum(), 2))

print("Average Order Value:", round(
    clean_df.groupby('Invoice')['Revenue'].sum().mean(), 2
))

Total Transactions: 36969
Total Customers: 5878
Total Products: 4631
Total Countries: 41
Total Revenue: 17374804.27
Average Order Value: 469.98


RFM CALCULATION

SET THE REFERENCE DATE

In [57]:
# Find the latest invoice date
latest_date = clean_df['InvoiceDate'].max()

# Set reference date
reference_date = latest_date + pd.Timedelta(days=1)

print("Latest Transaction Date:", latest_date)
print("Reference Date:", reference_date)

Latest Transaction Date: 2011-12-09 12:50:00
Reference Date: 2011-12-10 12:50:00


CALCULATE RFM VALUES

In [58]:
# Calculate RFM metrics
rfm = clean_df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'Invoice': 'nunique',
    'Revenue': 'sum'
})

# Rename columns
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'Invoice': 'Frequency',
    'Revenue': 'Monetary'
}, inplace=True)

# Display RFM table
rfm.head(10)

,Recency,Frequency,Monetary
Customer ID,,,
12346,326,12,77556.46
12347,2,8,4921.53
12348,75,5,2019.40
12349,19,4,4428.69
12350,310,1,334.40
12351,375,1,300.93
12352,36,10,2849.84
12353,204,2,406.76
12354,232,1,1079.40


CHECK RFM DATASET

In [59]:
print("Total Customers:", rfm.shape[0])

print("\nRFM Dataset Information:")
print(rfm.info())

print("\nRFM Statistical Summary:")
display(rfm.describe())

Total Customers: 5878

RFM Dataset Information:
<class 'pandas.DataFrame'>
Index: 5878 entries, 12346 to 18287
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Recency    5878 non-null   int64  
 1   Frequency  5878 non-null   int64  
 2   Monetary   5878 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 183.7 KB
None

RFM Statistical Summary:


,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000
mean,201.331916,6.289384,2955.904095
std,209.338707,13.009406,14440.852688
min,1.000000,1.000000,2.950000
25%,26.000000,1.000000,342.280000
50%,96.000000,3.000000,867.740000
75%,380.000000,7.000000,2248.305000
max,739.000000,398.000000,580987.040000


CHECK INDIVIDUAL CUSTOMER RECORDS

In [60]:
rfm.sort_values(
    by='Monetary',
    ascending=False
).head(10)

,Recency,Frequency,Monetary
Customer ID,,,
18102,1,145,580987.04
14646,2,151,528602.52
14156,10,156,313437.62
14911,1,398,291420.81
17450,8,51,244784.25
13694,4,143,195640.69
17511,3,60,172132.87
16446,1,2,168472.50
16684,4,55,147142.77


RFM SCORING

CALCULATE RFM SCORES

In [61]:
# Calculate RFM scores using quintiles

rfm['R_Score'] = pd.qcut(
    rfm['Recency'].rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

display(rfm.head(10))

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
Customer ID,,,,,,
12346,326,12,77556.46,2,5,5
12347,2,8,4921.53,5,4,5
12348,75,5,2019.40,3,4,4
12349,19,4,4428.69,5,3,5
12350,310,1,334.40,2,1,2
12351,375,1,300.93,2,1,2
12352,36,10,2849.84,4,5,4
12353,204,2,406.76,2,2,2
12354,232,1,1079.40,2,1,3


CREATE RFM SCORE

In [62]:
# Combine RFM scores
rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

display(rfm.head(10))

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
Customer ID,,,,,,,
12346,326,12,77556.46,2,5,5,255
12347,2,8,4921.53,5,4,5,545
12348,75,5,2019.40,3,4,4,344
12349,19,4,4428.69,5,3,5,535
12350,310,1,334.40,2,1,2,212
12351,375,1,300.93,2,1,2,212
12352,36,10,2849.84,4,5,4,454
12353,204,2,406.76,2,2,2,222
12354,232,1,1079.40,2,1,3,213


CREATE TOTAL RFM SCORE

In [63]:
rfm['RFM_Total'] = (
    rfm['R_Score'] +
    rfm['F_Score'] +
    rfm['M_Score']
)

display(rfm.head(10))

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total
Customer ID,,,,,,,,
12346,326,12,77556.46,2,5,5,255,12
12347,2,8,4921.53,5,4,5,545,14
12348,75,5,2019.40,3,4,4,344,11
12349,19,4,4428.69,5,3,5,535,13
12350,310,1,334.40,2,1,2,212,5
12351,375,1,300.93,2,1,2,212,5
12352,36,10,2849.84,4,5,4,454,13
12353,204,2,406.76,2,2,2,222,6
12354,232,1,1079.40,2,1,3,213,6


CHECK RFM SCORE DISTRIBUTION

In [64]:
print("RFM Score Distribution")

print("\nRecency Scores:")
print(rfm['R_Score'].value_counts().sort_index())

print("\nFrequency Scores:")
print(rfm['F_Score'].value_counts().sort_index())

print("\nMonetary Scores:")
print(rfm['M_Score'].value_counts().sort_index())

RFM Score Distribution

Recency Scores:
R_Score
1    1176
2    1175
3    1176
4    1175
5    1176
Name: count, dtype: int64

Frequency Scores:
F_Score
1    1176
2    1175
3    1176
4    1175
5    1176
Name: count, dtype: int64

Monetary Scores:
M_Score
1    1176
2    1175
3    1176
4    1175
5    1176
Name: count, dtype: int64


DISPLAY TOP 10 CUSTOMERS

In [65]:
top_customers = rfm.sort_values(
    by='RFM_Total',
    ascending=False
).head(10)

display(top_customers)

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total
Customer ID,,,,,,,,
14572,3,11,5526.880,5,5,5,555,15
17338,17,17,11889.330,5,5,5,555,15
13246,18,11,3351.750,5,5,5,555,15
16818,11,11,3349.570,5,5,5,555,15
16814,14,36,11924.850,5,5,5,555,15
15750,3,18,6452.991,5,5,5,555,15
13225,3,16,16260.070,5,5,5,555,15
15712,9,12,3472.410,5,5,5,555,15
15738,19,41,13043.320,5,5,5,555,15


CUSTOMER SEGMENTATION

In [66]:
# Customer Segmentation

def customer_segment(row):

    if row['R_Score'] >= 4 and row['F_Score'] >= 4:
        return 'Champions'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal Customers'

    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New Customers'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At-Risk Customers'

    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'Lost Customers'

    else:
        return 'Potential Loyalists'


rfm['Customer_Segment'] = rfm.apply(
    customer_segment,
    axis=1
)

display(rfm.head(10))

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total,Customer_Segment
Customer ID,,,,,,,,,
12346,326,12,77556.46,2,5,5,255,12,At-Risk Customers
12347,2,8,4921.53,5,4,5,545,14,Champions
12348,75,5,2019.40,3,4,4,344,11,Loyal Customers
12349,19,4,4428.69,5,3,5,535,13,Loyal Customers
12350,310,1,334.40,2,1,2,212,5,Lost Customers
12351,375,1,300.93,2,1,2,212,5,Lost Customers
12352,36,10,2849.84,4,5,4,454,13,Champions
12353,204,2,406.76,2,2,2,222,6,Lost Customers
12354,232,1,1079.40,2,1,3,213,6,Lost Customers


FINAL DATSET FOR POWERBI

In [67]:
# Prepare final customer segmentation dataset

final_df = rfm.reset_index()

# Save final dataset
final_df.to_csv(
    'Customer_Segmentation_Results.csv',
    index=False
)

print("Customer Segmentation Dataset Created Successfully!")

display(final_df.head())

Customer Segmentation Dataset Created Successfully!


,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total,Customer_Segment
0,12346,326,12,77556.46,2,5,5,255,12,At-Risk Customers
1,12347,2,8,4921.53,5,4,5,545,14,Champions
2,12348,75,5,2019.40,3,4,4,344,11,Loyal Customers
3,12349,19,4,4428.69,5,3,5,535,13,Loyal Customers
4,12350,310,1,334.40,2,1,2,212,5,Lost Customers


# 10. BUSINESS RECOMMENDATIONS & STRATEGIC INSIGHTS

## 10.1 Introduction

The RFM (Recency, Frequency, Monetary) analysis provides valuable insights into customer purchasing behaviour by evaluating how recently customers purchased, how frequently they purchase, and how much they spend. Based on these metrics, customers are classified into six segments.

The following strategic recommendations are proposed to improve customer engagement, strengthen retention, and maximize business revenue.

## 10.2 Segment-Wise Business Strategies

### 1. Champions
**Business Objective:** Retain high-value customers and strengthen brand loyalty.

- Introduce exclusive membership and loyalty reward programs.
- Provide early access to new products and promotional offers.
- Encourage referrals through customer advocacy programs.
- Offer personalized recommendations based on purchase history.

### 2. Loyal Customers
**Business Objective:** Increase purchase frequency and customer lifetime value.

- Introduce tier-based loyalty programs.
- Provide personalized discounts and product recommendations.
- Encourage repeat purchases through reward points.
- Promote complementary and related products.

### 3. Potential Loyalists
**Business Objective:** Convert promising customers into long-term loyal customers.

- Offer incentives for their next purchase.
- Recommend products based on previous transactions.
- Introduce personalized marketing campaigns.
- Encourage participation in customer loyalty programs.

### 4. New Customers
**Business Objective:** Improve customer onboarding and encourage repeat purchases.

- Provide welcome discounts and introductory offers.
- Share product information and purchasing guidance.
- Send personalized follow-up communications.
- Offer incentives for a second purchase.

### 5. At-Risk Customers
**Business Objective:** Reduce customer inactivity and prevent customer churn.

- Launch targeted customer re-engagement campaigns.
- Provide limited-period promotional offers.
- Identify changes in purchasing behaviour.
- Send personalized reminders and product recommendations.

### 6. Lost Customers
**Business Objective:** Reactivate inactive customers where commercially viable.

- Develop targeted win-back campaigns.
- Offer special discounts on selected products.
- Communicate new product launches and improvements.
- Evaluate reactivation campaign performance before investing further.

## 10.3 Expected Business Benefits

Implementing RFM-based customer segmentation can support the following business outcomes:

| Business Area | Expected Benefit |
|---|---|
| Customer Retention | Improved customer relationships and loyalty |
| Revenue Growth | Increased repeat purchases and sales opportunities |
| Marketing Efficiency | More targeted promotional campaigns |
| Customer Engagement | Personalized communication and recommendations |
| Customer Churn | Early identification of inactive and at-risk customers |
| Business Decision-Making | Data-driven customer management strategies |

## 10.4 Strategic Conclusion

RFM-based customer segmentation provides a structured approach to understanding customer purchasing patterns and identifying different levels of customer value.

By implementing segment-specific marketing strategies, e-commerce businesses can improve customer retention, encourage repeat purchases, optimize promotional activities, and identify opportunities for revenue growth.

The analysis demonstrates how data analytics can transform transactional data into meaningful business insights and support informed managerial decision-making.